# 🧠 Mixture of Experts (MoE) Router - Groq API
Unit 2 Assignment

This notebook implements a Smart Customer Support Router using a Mixture of Experts architecture.

## 1️⃣ Install Dependencies

In [13]:
!pip install groq python-dotenv

## 2️⃣ Import Libraries and Setup API

In [27]:
import os
from groq import Groq
from dotenv import load_dotenv

# Load environment variables (optional, as key will be hardcoded)
load_dotenv()


client = Groq(api_key="GROQ_API_KEY")

BASE_MODEL = "llama-3.1-8b-instant"

## 3️⃣ Define Expert Configurations

In [15]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert.
Be precise, logical, and code-focused.
Explain errors clearly and provide corrected code when possible.
Avoid unnecessary fluff."""
    },
    "billing": {
        "system_prompt": """You are a Billing Support Specialist.
Be empathetic and professional.
Focus on subscriptions, refunds, and payment policies.
Provide clear next steps."""
    },
    "general": {
        "system_prompt": """You are a friendly and helpful general assistant.
Answer clearly and conversationally."""
    },
    "tool": {
        "system_prompt": "TOOL_HANDLER"
    }
}


## 4️⃣ Router Function

In [16]:
def route_prompt(user_input: str) -> str:

    router_prompt = f"""
Classify the following user query into one of these categories:
[technical, billing, general, tool]

Rules:
- technical → bugs, errors, programming issues
- billing → payments, refunds, charges, subscriptions
- tool → requests for real-time data (e.g., current price of Bitcoin)
- general → anything else

Return ONLY the category word.

User Query:
{user_input}
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": "You are a strict classifier."},
            {"role": "user", "content": router_prompt}
        ],
        temperature=0
    )

    category = response.choices[0].message.content.strip().lower()

    if category not in MODEL_CONFIG:
        return "general"

    return category


## 5️⃣ Tool Handler (Bonus)

In [17]:
def get_bitcoin_price():
    # Mock data (replace with real API if needed)
    return "Bitcoin is currently trading at $63,500 (mock data)."


## 6️⃣ Orchestrator

In [18]:
def process_request(user_input: str):

    category = route_prompt(user_input)
    print(f"[Router Decision]: {category}")

    if category == "tool":
        if "bitcoin" in user_input.lower():
            return get_bitcoin_price()
        else:
            return "Tool request recognized but not implemented."

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    return response.choices[0].message.content


## 7️⃣ Test the Router

In [28]:
# Example test
query = "My python script is throwing an IndexError on line 5."
print(process_request(query))

[Router Decision]: technical
To troubleshoot the issue, I'll need more information about your code. Please provide the following:

1. The exact error message you're seeing.
2. The relevant part of your code, specifically lines 1-10.
3. Any relevant variables or data structures used in your code.

However, I can provide a general explanation of what an IndexError typically means:

**IndexError:**

An IndexError occurs when you're trying to access an element in a sequence (like a list, tuple, or string) using an index that's out of range. In Python, list indices start at 0, so if you're trying to access an element at index 5 in a list with only 4 elements, you'll get an IndexError.

To fix this issue, you can:

1. Check the length of your list or string to ensure it's not empty.
2. Verify that the index you're using is within the valid range.
3. Use the `len()` function to get the length of your sequence and use that to access the last element.

Example:
```python
my_list = [1, 2, 3]
pri

In [29]:
query = "I want to know why I was charged twice for my subscription this month."
print(process_request(query))

[Router Decision]: billing
I'm here to help you resolve the issue. I apologize for the inconvenience you've experienced. Being charged twice for a subscription can be frustrating, and I'm happy to assist you in understanding what might have caused this.

To better investigate this issue, could you please provide me with some information? 

1. What is your subscription name/plan?
2. Can you tell me the dates when the duplicate charges appeared on your statement?
3. Did you make any changes to your subscription or account details around the time of the duplicate charges?
4. Have you received any notifications or alerts from us regarding these charges?

Once I have this information, I'll be able to check our system and determine what might have caused the duplicate charge. I'll then guide you through the next steps to get this resolved for you.

Please know that we take these types of issues seriously, and I'm committed to making things right.
